# Run17 Expanded Universe Training (Local)
Thin orchestration notebook for the canonical Run17 expanded-universe training path.
20-asset universe with 10-year train (2010-2019) / 5-year test (2020-2024) split.
Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/environment source files.

## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [ ]:
import shutil
from pathlib import Path
repo = Path("/content/tcn_tape_vectorized_version_clean")
if repo.exists():
    shutil.rmtree(repo, ignore_errors=True)
print("removed", repo)

In [ ]:
import gc
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

# GitHub-native setup knobs.
GIT_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
GIT_BRANCH = "feature/run17-film-drive-org-20260318"    # e.g. 'feature/run17-film-drive-org-20260318'
CLONE_IF_MISSING = True
CLONE_PARENT_DIR = Path('/content')
CLONE_DIR_NAME = 'tcn_tape_vectorized_version_clean'

# Notebook/runtime knobs.
TRAIN_BRANCH = None  # Optional post-clone/post-open checkout override.
INSTALL_REQUIREMENTS = True
AUTO_INSTALL_MISSING_REQUIREMENTS = True
RESET_OUTPUT_DIRS = True


def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)


CRITICAL_RUNTIME_MODULES = {
    'pandas_ta_classic': 'pandas-ta-classic>=0.3.59',
    'fredapi': 'fredapi>=0.5.1',
    'yfinance': 'yfinance>=0.2.38',
}


def missing_runtime_requirements() -> list[str]:
    missing = []
    for module_name, requirement in CRITICAL_RUNTIME_MODULES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(requirement)
    return missing


def normalize_github_url(url: str | None) -> str | None:
    if not url:
        return url
    url = str(url).strip()
    if url.startswith('git@github.com:'):
        repo = url[len('git@github.com:'):]
        if repo.endswith('.git'):
            repo = repo[:-4]
        return f'https://github.com/{repo}.git'
    return url


def find_repo_root() -> Path | None:
    candidate_roots = []
    seen = set()
    for p in [Path.cwd(), *Path.cwd().parents]:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)
    for p in [
        CLONE_PARENT_DIR / CLONE_DIR_NAME,
        Path('/content/repo'),
        Path('/content/project'),
        Path('C:/Users/Owner/tcn_tape_vectorized_version_clean'),
        Path('/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean'),
    ]:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)

    for p in candidate_roots:
        if (p / '.git').exists() and (p / 'src').exists() and (p / 'tcn_architecture_analysis.ipynb').exists():
            return p
    return None


TRAIN_REPO_DIR = find_repo_root()
GIT_REPO_URL = normalize_github_url(GIT_REPO_URL)

if TRAIN_REPO_DIR is None and CLONE_IF_MISSING:
    if not GIT_REPO_URL:
        raise FileNotFoundError(
            'Repo root not found and GIT_REPO_URL is not set. '
            'Set GIT_REPO_URL to your GitHub clone URL for a fresh runtime.'
        )
    CLONE_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    clone_target = CLONE_PARENT_DIR / CLONE_DIR_NAME
    if clone_target.exists() and not (clone_target / '.git').exists():
        print(f'[WARN] Removing partial non-git clone target: {clone_target}')
        shutil.rmtree(clone_target, ignore_errors=True)
    if not clone_target.exists():
        try:
            run(['git', 'clone', GIT_REPO_URL, str(clone_target)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                'git clone failed. Use an HTTPS GitHub URL in GIT_REPO_URL. '                f'Normalized URL: {GIT_REPO_URL}'
            ) from exc
    TRAIN_REPO_DIR = clone_target.resolve()

if TRAIN_REPO_DIR is None:
    attempted = '\n'.join([
        f' - {Path.cwd().resolve()}',
        *[f' - {p.resolve()}' for p in Path.cwd().parents],
        f' - {(CLONE_PARENT_DIR / CLONE_DIR_NAME).resolve()}',
        ' - /content/repo',
        ' - /content/project',
        ' - C:/Users/Owner/tcn_tape_vectorized_version_clean',
        ' - /mnt/c/Users/Owner/tcn_tape_vectorized_version_clean',
    ])
    raise FileNotFoundError('Repo root not found. Tried:\n' + attempted)

REQUESTED_BRANCH = TRAIN_BRANCH or GIT_BRANCH
if REQUESTED_BRANCH:
    run(['git', '-C', str(TRAIN_REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(TRAIN_REPO_DIR), 'checkout', REQUESTED_BRANCH])
    # In Colab, prefer the exact remote branch tip over any stale local branch state.
    try:
        run(['git', '-C', str(TRAIN_REPO_DIR), 'reset', '--hard', f'origin/{REQUESTED_BRANCH}'])
    except subprocess.CalledProcessError:
        print(f'[WARN] No origin tracking ref found for {REQUESTED_BRANCH}; using local checkout only.')

if RESET_OUTPUT_DIRS:
    purge_paths = [
        TRAIN_REPO_DIR / 'tcn_fusion_results',
        TRAIN_REPO_DIR / 'tcn_results',
        TRAIN_REPO_DIR / 'tcn_att_results',
        TRAIN_REPO_DIR / 'results',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data' / 'daily_ohlcv_assets.csv',
        TRAIN_REPO_DIR / 'data' / 'processed_daily_macro_features.csv',
    ]
    for path in purge_paths:
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        elif path.exists():
            path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob('.ipynb_checkpoints'):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == 'src' or mod.startswith('src.'):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

requirements_file = TRAIN_REPO_DIR / 'requirements.txt'
missing_requirements = missing_runtime_requirements()
should_install = INSTALL_REQUIREMENTS or (AUTO_INSTALL_MISSING_REQUIREMENTS and bool(missing_requirements))
if should_install:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    if INSTALL_REQUIREMENTS:
        run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    elif missing_requirements:
        run([sys.executable, '-m', 'pip', 'install', *missing_requirements])

print('[OK] Repo ready:', TRAIN_REPO_DIR)
print('[OK] Normalized GIT_REPO_URL:', GIT_REPO_URL)
run(['git', '-C', str(TRAIN_REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'])
run(['git', '-C', str(TRAIN_REPO_DIR), 'rev-parse', 'HEAD'])
print('[OK] Requirements install requested:', INSTALL_REQUIREMENTS)
print('[OK] Missing critical requirements before install:', missing_requirements)
print('[OK] Dependency install executed:', should_install)
print('[OK] GitHub clone-if-missing:', CLONE_IF_MISSING)
print('[OK] Requested git branch:', GIT_BRANCH)


In [ ]:
import tensorflow as tf
import subprocess

REQUIRE_GPU = False  # set True if you want hard fail without GPU

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if smi.returncode == 0:
        print("nvidia-smi:", [line.strip() for line in smi.stdout.splitlines() if line.strip()])
    else:
        print("nvidia-smi: not available")
except Exception:
    print("nvidia-smi: not available")

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)

if not gpus:
    msg = 'No GPU visible to TensorFlow; running on CPU (slower).'
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('[WARN]', msg)
else:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
print('TF build CUDA:', tf.test.is_built_with_cuda())


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run18_config, assert_run18_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import (
    _get_results_root_for_architecture,
    prepare_phase1_dataset,
    run_experiment6_tape,
)

RUN_ID = 'run18'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None
RUN_CONFIG_LINEAGE = 'run18'


In [ ]:
import logging

QUIET_SRC_INFO_LOGS = True

if QUIET_SRC_INFO_LOGS:
    # Suppress noisy module INFO logs like: "... - src.environment_tape_rl - INFO ..."
    for logger_name in [
        "src",
        "src.environment_tape_rl",
        "src.notebook_helpers.tcn_phase1",
        "src.agents.ppo_agent_tf",
    ]:
        logging.getLogger(logger_name).setLevel(logging.WARNING)
    print("[OK] Suppressed src INFO logs (level=WARNING).")
else:
    print("[INFO] Src INFO logs left enabled.")


## 3) Build Canonical Run18 Config and Dataset
Create the source-backed Run18 config, assert no drift, and prepare the dataset once.


In [ ]:
train_config = build_run18_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run18_config(train_config)

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']
feature_params = train_config.get('feature_params', {})
feature_selection = feature_params.get('feature_selection', {}) if isinstance(feature_params.get('feature_selection', {}), dict) else {}

print('[Run18] Config ready for training')
print('  run_id =', RUN_ID)
print('  config_lineage =', RUN_CONFIG_LINEAGE)
print('  num_assets =', train_config['NUM_ASSETS'])
print('  tickers =', train_config['ASSET_TICKERS'])
print('  analysis_start_date =', train_config['ANALYSIS_START_DATE'])
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  max_total_timesteps =', tp['max_total_timesteps'])
print('  tcn_filters =', ap.get('tcn_filters'))
print('  tcn_dilations =', ap.get('tcn_dilations'))
print('  tcn_kernel_size =', ap.get('tcn_kernel_size'))
print('  dirichlet_alpha_activation =', ap.get('dirichlet_alpha_activation'))
print('  dirichlet_softplus_alpha_floor =', ap.get('dirichlet_softplus_alpha_floor'))
print('  dirichlet_softplus_alpha_scale =', ap.get('dirichlet_softplus_alpha_scale'))
print('  dirichlet_cross_sectional_standardize =', ap.get('dirichlet_cross_sectional_standardize'))
print('  lagrangian_cvar_enabled =', ppo['lagrangian_cvar_enabled'])
print('  alpha_regularization =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'target_std': ppo['alpha_dispersion_target_std'],
})
print('  entropy_schedule =', tp['ppo_entropy_coef_schedule'])
print('  beta_curriculum =', tp['action_execution_beta_curriculum'])
print('  turnover_curriculum =', tp['turnover_penalty_curriculum'])
print('  reward_component_schedule =', tp['reward_component_schedule'])
print('  episode_length_curriculum =', tp['episode_length_curriculum_schedule'])
print('  feature_allowlist_count =', len(feature_selection.get('active_features_allowlist', []) or []))

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

observed_tickers = sorted(set(train_phase1_data.master_df['Ticker'].astype(str))) if 'Ticker' in train_phase1_data.master_df.columns else []
missing_tickers = sorted(set(train_config['ASSET_TICKERS']) - set(observed_tickers))
if missing_tickers:
    raise RuntimeError(f'Missing Run18 asset tickers in prepared dataset: {missing_tickers}')

requested_cols = list(train_phase1_data.data_processor.get_feature_columns('phase1'))
active_cols = [col for col in requested_cols if col in train_phase1_data.master_df.columns]
explicit_global = list(env.get('global_feature_columns', []) or [])
explicit_prefixes = list(env.get('global_feature_prefixes', []) or [])

global_set = set()
for col in active_cols:
    if col in explicit_global or any(col.startswith(prefix) for prefix in explicit_prefixes):
        global_set.add(col)

if 'Date' in train_phase1_data.master_df.columns:
    grouped = train_phase1_data.master_df.groupby('Date')
    for col in active_cols:
        if col in global_set:
            continue
        try:
            nunique_per_date = grouped[col].nunique(dropna=True)
        except Exception:
            continue
        if len(nunique_per_date) > 0 and int(nunique_per_date.max()) <= 1:
            global_set.add(col)

global_feature_columns = [col for col in active_cols if col in global_set]
asset_feature_columns = [col for col in active_cols if col not in global_set]

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Observed tickers:', observed_tickers)
print('[OK] Active feature count:', len(active_cols))
print('[OK] Asset feature count per asset:', len(asset_feature_columns))
print('[OK] Global feature count:', len(global_feature_columns))
print('[OK] Asset feature columns:', asset_feature_columns)
print('[OK] Global feature columns:', global_feature_columns)


## 4) Run Training
Launch the canonical Run18 training path.


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [ ]:
TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
    architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
    use_attention=bool(train_config['agent_params'].get('use_attention', False)),
    use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
    project_root=TRAIN_REPO_DIR,
)
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

print('Results root:', TRAIN_RESULTS_ROOT)
print('Logs dir:', TRAIN_LOGS_DIR)

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


## 6) Export Artifacts (Optional)
Zip the latest results and save them into a dedicated Google Drive folder for this run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import subprocess

EXPORT_RESULTS_ZIP = True
SAVE_TO_DRIVE = True
DRIVE_EXPORT_ROOT = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs')
DRIVE_EXPORT_DIR = DRIVE_EXPORT_ROOT / RUN_ID
LOCAL_EXPORT_PATH = TRAIN_REPO_DIR / f"tcn_tape_vectorized_{RUN_ID}.zip"

if 'TRAIN_RESULTS_ROOT' not in globals():
    TRAIN_RESULTS_ROOT = _get_results_root_for_architecture(
        architecture=train_config['agent_params'].get('actor_critic_type', 'TCN_FUSION'),
        use_attention=bool(train_config['agent_params'].get('use_attention', False)),
        use_fusion=bool(train_config['agent_params'].get('use_fusion', False)),
        project_root=TRAIN_REPO_DIR,
    )

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_RESULTS_ROOT,
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if LOCAL_EXPORT_PATH.exists():
            LOCAL_EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            [
                'bash',
                '-lc',
                'cd "{}" && zip -qr "{}" {}'.format(
                    TRAIN_REPO_DIR,
                    LOCAL_EXPORT_PATH,
                    ' '.join(f'"{item}"' for item in relative_items),
                ),
            ],
            check=True,
        )
        print('[OK] Created local zip:', LOCAL_EXPORT_PATH)

        if SAVE_TO_DRIVE:
            if not Path('/content/drive/MyDrive').exists():
                raise FileNotFoundError('Google Drive is not mounted at /content/drive/MyDrive')
            DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
            drive_zip_path = DRIVE_EXPORT_DIR / LOCAL_EXPORT_PATH.name
            shutil.copy2(LOCAL_EXPORT_PATH, drive_zip_path)
            print('[OK] Copied zip to Drive:', drive_zip_path)
            print('[OK] Drive run folder:', DRIVE_EXPORT_DIR)
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')
    print('Drive export folder would be:', DRIVE_EXPORT_DIR)
